In [34]:
import pandas as pd
from utils import drop_rows, drop_columns
import numpy as np

In [ ]:
file="../bronze/weather/weather.parquet"

In [ ]:
df = pd.read_parquet(file)
#df
df.info()

DELETE ALL THE DATA BEFORE AND AFTER THE YEAR 2013

In [ ]:
new_df= drop_rows(df,'date')
new_df
print(new_df.columns)

INVESTIGATE WHICH TYPE OF DATA INCLUDES THE OBJECT TYPE

In [ ]:
print(df.select_dtypes(include="object").dtypes)
for col in df.select_dtypes(include="object").columns:
    print(f"{col}: {df[col].dropna().head(1).tolist()}")

DELETE UNNECESSARY COLUMNS FOR THE ANALYSIS

In [ ]:
index_delete_columns=new_df.columns[16:59]
name_delete_columns=['latitude','longitude','elevation']

first_delete_columns=drop_columns(new_df, index_delete_columns)
second_delete_columns=drop_columns(first_delete_columns, name_delete_columns)

second_delete_columns.head()

GET ATTRIBUTES THAT INDICATED THE QUALITY OF INFORMATION

In [ ]:
# Select all attribute columns
attribute_columns = [
    col for col in second_delete_columns.columns
    if col.endswith("_attributes")
]

# Extract quality flags from all attribute columns
quality_flags = second_delete_columns[attribute_columns].apply(
    lambda col: col.str.split(",", n=1).str[0].mask(lambda s: s == "")
)

#Where the first flag is not empyty
has_flag = quality_flags.notna()

#Get the rows where the flag is not empty
rows_with_issues = second_delete_columns.loc[has_flag.any(axis=1)]
print(rows_with_issues)


           station        date                         name  prcp  \
56493  USW00094728  2023-09-04  NY CITY CENTRAL PARK, NY US     0   
56496  USW00094728  2023-09-07  NY CITY CENTRAL PARK, NY US     0   
56528  USW00094728  2023-10-09  NY CITY CENTRAL PARK, NY US     0   
56535  USW00094728  2023-10-16  NY CITY CENTRAL PARK, NY US     0   
56550  USW00094728  2023-10-31  NY CITY CENTRAL PARK, NY US     0   
56555  USW00094728  2023-11-05  NY CITY CENTRAL PARK, NY US     0   
56563  USW00094728  2023-11-13  NY CITY CENTRAL PARK, NY US     0   
56568  USW00094728  2023-11-18  NY CITY CENTRAL PARK, NY US     0   
56578  USW00094728  2023-11-28  NY CITY CENTRAL PARK, NY US     0   
56587  USW00094728  2023-12-07  NY CITY CENTRAL PARK, NY US     0   
56589  USW00094728  2023-12-09  NY CITY CENTRAL PARK, NY US     0   

      prcp_attributes  snow snow_attributes  snwd snwd_attributes   tmax  ...  \
56493       T,,W,2400   0.0        ,,W,2400   0.0        ,,W,2400  317.0  ...   
56496    

In [ ]:
# Select rows with at least one non-empty quality flag
not_empty_data = second_delete_columns[
    quality_flags.notna().any(axis=1)
    & quality_flags.ne("").any(axis=1)
]

# Identify non-empty quality flags
has_quality_issue = quality_flags.notna() & quality_flags.ne("")

# Keep rows with at least one quality issue
not_empty_data = second_delete_columns[
    has_quality_issue.any(axis=1)
]

#not_empty_data